In [1]:
import os
import sys
import json
import time
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join(os.getcwd(), 'implementation'))
class Config:
    DATASET_PATH = os.path.join(os.getcwd(), "dataset", "AugmentedAlzheimerDataset")
    OUTPUT_PATH = os.path.join(os.getcwd(), "quantum_features")
    FEDERATED_DATA_PATH = os.path.join(os.getcwd(), "federated_data_subset")
    
    IMG_SIZE = 64
    PATCH_SIZE = 2
    NUM_QUBITS = 4
    
    NUM_CLIENTS = 5
    SAMPLES_PER_CLIENT = 2000
    
    QUANTUM_CIRCUIT_DEPTH = 2
    
    DISTRIBUTION_RATIOS = {
        1: {"MildDemented": 0.25, "ModerateDemented": 0.25, "NonDemented": 0.25, "VeryMildDemented": 0.25},
        2: {"MildDemented": 0.25, "ModerateDemented": 0.25, "NonDemented": 0.25, "VeryMildDemented": 0.25},
        3: {"MildDemented": 0.25, "ModerateDemented": 0.25, "NonDemented": 0.25, "VeryMildDemented": 0.25},
        4: {"MildDemented": 0.25, "ModerateDemented": 0.25, "NonDemented": 0.25, "VeryMildDemented": 0.25},
        5: {"MildDemented": 0.25, "ModerateDemented": 0.25, "NonDemented": 0.25, "VeryMildDemented": 0.25},
    }
    
    CLASSES = ["MildDemented", "ModerateDemented", "NonDemented", "VeryMildDemented"]
config = Config()
print("Configuration loaded:")
print(f"  Dataset: {config.DATASET_PATH}")
print(f"  Output: {config.OUTPUT_PATH}")
print(f"  Images per client: {config.SAMPLES_PER_CLIENT}")
print(f"  Feature dimensions: {(config.IMG_SIZE // config.PATCH_SIZE) ** 2 * config.NUM_QUBITS}")

Configuration loaded:
  Dataset: c:\Users\prakh\Downloads\VIT Downloads\6th Sem\Project\Alzheimer-s-Detection-Using-Quanvolutional-Neural-Networks-with-Federated-Training\dataset\AugmentedAlzheimerDataset
  Output: c:\Users\prakh\Downloads\VIT Downloads\6th Sem\Project\Alzheimer-s-Detection-Using-Quanvolutional-Neural-Networks-with-Federated-Training\quantum_features
  Images per client: 2000
  Feature dimensions: 4096


In [2]:
def get_class_images(source_path, classes):
    class_images = {}
    for cls in classes:
        cls_path = os.path.join(source_path, cls)
        if os.path.exists(cls_path):
            files = [f for f in os.listdir(cls_path) if f.endswith((".jpg", ".png", ".jpeg"))]
            class_images[cls] = files
            print(f"  {cls}: {len(files)} images")
        else:
            print(f"  {cls}: NOT FOUND")
            class_images[cls] = []
    return class_images
def load_image(image_path):
    img = Image.open(image_path).convert('L')
    img = img.resize((64, 64))
    img_array = np.array(img) / 255.0  # NORMALIZE TO 0-1 range (critical for quantum circuit!)
    return img_array
def create_non_iid_subset(class_images, config):
    client_data = {i: [] for i in range(1, config.NUM_CLIENTS + 1)}
    
    for client_id in range(1, config.NUM_CLIENTS + 1):
        ratios = config.DISTRIBUTION_RATIOS[client_id]
        target_total = config.SAMPLES_PER_CLIENT
        
        for cls in config.CLASSES:
            available = len(class_images[cls])
            
            if target_total is None:
                n_samples = int(available * ratios[cls])
            else:
                n_samples = int(target_total * ratios[cls])
            
            n_samples = min(n_samples, available)
            selected = random.sample(class_images[cls], n_samples) if n_samples <= available else class_images[cls][:n_samples]
            
            for filename in selected:
                client_data[client_id].append({
                    'image_path': os.path.join(config.DATASET_PATH, cls, filename),
                    'label': cls,
                    'label_idx': config.CLASSES.index(cls),
                    'filename': filename
                })
        
        random.shuffle(client_data[client_id])
        print(f"Client {client_id}: {len(client_data[client_id])} images")
    
    return client_data
print("Loading dataset info...")
class_images = get_class_images(config.DATASET_PATH, config.CLASSES)

Loading dataset info...
  MildDemented: 8960 images
  ModerateDemented: 6464 images
  NonDemented: 9600 images
  VeryMildDemented: 8960 images


In [3]:
import pennylane as qml
import numpy as np
from config import NUM_QUBITS, IMG_SIZE, PATCH_SIZE

dev = qml.device("default.qubit", wires=NUM_QUBITS)


@qml.qnode(dev)
def quantum_patch_circuit(phi):
    """
    Quantum circuit for feature extraction from image patches.

    Args:
        phi: Input parameters (pixel values normalized to rotations)

    Returns:
        List of Pauli-Z expectation values
    """
    # RX rotations on each qubit
    qml.RX(phi[0], wires=0)
    qml.RX(phi[1], wires=1)
    qml.RX(phi[2], wires=2)
    qml.RX(phi[3], wires=3)

    # CNOT entanglement layers
    qml.CNOT(wires=[0, 1])
    qml.RZ(np.pi / 2, wires=1)
    qml.CNOT(wires=[0, 1])

    qml.CNOT(wires=[2, 3])
    qml.RZ(np.pi / 2, wires=3)
    qml.CNOT(wires=[2, 3])

    qml.CNOT(wires=[1, 2])
    qml.RZ(np.pi / 2, wires=2)
    qml.CNOT(wires=[1, 2])

    # Return Pauli-Z expectation values
    return [qml.expval(qml.PauliZ(i)) for i in range(NUM_QUBITS)]


def extract_quantum_features(image, patch_size=PATCH_SIZE, img_size=IMG_SIZE):
    """
    Extract quantum features from an image using patch-based processing.

    Args:
        image: Grayscale image (img_size x img_size)
        patch_size: Size of patches to process (default 2x2)
        img_size: Image size (default 64)

    Returns:
        Flattened numpy array of quantum features
    """
    h, w = image.shape
    features = []

    for i in range(0, h - patch_size + 1, patch_size):
        for j in range(0, w - patch_size + 1, patch_size):
            patch = image[i : i + patch_size, j : j + patch_size].flatten()

            # Ensure patch has enough values for all qubits
            if len(patch) < NUM_QUBITS:
                patch = np.concatenate([patch, np.zeros(NUM_QUBITS - len(patch))])
            elif len(patch) > NUM_QUBITS:
                patch = patch[:NUM_QUBITS]

            # Normalize pixel values to [-π, π] for RX rotations
            normalized_patch = (patch * 2 - 1) * np.pi

            # Apply quantum circuit
            expectations = quantum_patch_circuit(normalized_patch)
            features.append(expectations)

    return np.array(features).flatten()
def extract_features_batch(image_paths, config, verbose=True):
    n_images = len(image_paths)
    feature_dim = (config.IMG_SIZE // config.PATCH_SIZE) ** 2 * config.NUM_QUBITS
    
    features = np.zeros((n_images, feature_dim), dtype=np.float32)
    labels = np.zeros(n_images, dtype=np.int32)
    extraction_times = []
    
    if verbose:
        iterator = tqdm(enumerate(image_paths), total=n_images, desc="Extracting features")
    else:
        iterator = enumerate(image_paths)
    
    for i, item in iterator:
        img = load_image(item['image_path'])
        
        start_time = time.time()
        feature_vector = extract_quantum_features(img, config.PATCH_SIZE, config.IMG_SIZE)
        elapsed = time.time() - start_time
        
        extraction_times.append(elapsed)
        features[i] = feature_vector.astype(np.float32)
        labels[i] = item['label_idx']
    
    metadata = {
        'total_images': n_images,
        'feature_dim': feature_dim,
        'mean_time_per_image': np.mean(extraction_times),
        'std_time_per_image': np.std(extraction_times),
        'total_time': np.sum(extraction_times),
    }
    
    return features, labels, metadata
print("Quantum feature extraction module loaded.")
print(f"Feature dimension: {(config.IMG_SIZE // config.PATCH_SIZE) ** 2 * config.NUM_QUBITS}")

Quantum feature extraction module loaded.
Feature dimension: 4096


In [4]:
def save_client_features(client_id, features, labels, config):
    client_folder = os.path.join(config.OUTPUT_PATH, f"client_{client_id}")
    os.makedirs(client_folder, exist_ok=True)
    
    np.save(os.path.join(client_folder, "features.npy"), features)
    np.save(os.path.join(client_folder, "labels.npy"), labels)
    
    client_config = {
        'client_id': client_id,
        'num_samples': len(features),
        'feature_dim': features.shape[1],
        'class_distribution': {}
    }
    
    for cls_idx in np.unique(labels):
        cls_name = config.CLASSES[cls_idx]
        count = np.sum(labels == cls_idx)
        client_config['class_distribution'][cls_name] = int(count)
    
    with open(os.path.join(client_folder, "config.json"), 'w') as f:
        json.dump(client_config, f, indent=2)
    
    return client_folder
def save_global_metadata(config, extraction_results):
    metadata = {
        'config': {
            'dataset_path': config.DATASET_PATH,
            'img_size': config.IMG_SIZE,
            'patch_size': config.PATCH_SIZE,
            'num_qubits': config.NUM_QUBITS,
            'num_clients': config.NUM_CLIENTS,
            'samples_per_client': config.SAMPLES_PER_CLIENT,
            'distribution': 'iid',
        },
        'extraction_results': extraction_results,
        'classes': config.CLASSES,
        'distribution_ratios': config.DISTRIBUTION_RATIOS
    }
    
    with open(os.path.join(config.OUTPUT_PATH, "metadata.json"), 'w') as f:
        json.dump(metadata, f, indent=2)
def load_client_features(client_id, config):
    client_folder = os.path.join(config.OUTPUT_PATH, f"client_{client_id}")
    features = np.load(os.path.join(client_folder, "features.npy"))
    labels = np.load(os.path.join(client_folder, "labels.npy"))
    return features, labels
print("Feature saver module loaded.")

Feature saver module loaded.


In [5]:
def run_full_extraction(config, verbose=True):
    start_total = time.time()
    
    if os.path.exists(config.OUTPUT_PATH):
        shutil.rmtree(config.OUTPUT_PATH)
    os.makedirs(config.OUTPUT_PATH)
    
    print("\n" + "="*60)
    print("STEP 1: Loading dataset...")
    print("="*60)
    class_images = get_class_images(config.DATASET_PATH, config.CLASSES)
    
    print("\n" + "="*60)
    print("STEP 2: Creating subsets...")
    print("="*60)
    client_data = create_non_iid_subset(class_images, config)
    
    print("\n" + "="*60)
    print("STEP 3: Extracting quantum features...")
    print("="*60)
    
    extraction_results = {}
    
    for client_id in range(1, config.NUM_CLIENTS + 1):
        print(f"\n  Processing Client {client_id}...")
        
        features, labels, metadata = extract_features_batch(
            client_data[client_id], config, verbose=verbose
        )
        
        client_folder = save_client_features(client_id, features, labels, config)
        
        extraction_results[client_id] = {
            'num_samples': len(features),
            'feature_dim': features.shape[1],
            'mean_extraction_time': metadata['mean_time_per_image'],
            'total_extraction_time': metadata['total_time'],
            'storage_size_mb': (features.nbytes + labels.nbytes) / (1024 * 1024)
        }
        
        print(f"  -> Saved {len(features)} features to {client_folder}")
        print(f"  -> Storage: {extraction_results[client_id]['storage_size_mb']:.2f} MB")
        print(f"  -> Avg time/image: {metadata['mean_time_per_image']:.3f}s")
    
    print("\n" + "="*60)
    print("STEP 4: Saving metadata...")
    print("="*60)
    save_global_metadata(config, extraction_results)
    
    total_time = time.time() - start_total
    
    print("\n" + "="*60)
    print("EXTRACTION COMPLETE!")
    print("="*60)
    print(f"Total time: {total_time/60:.1f} minutes")
    print(f"Output: {config.OUTPUT_PATH}")
    print(f"Total samples: {config.SAMPLES_PER_CLIENT * config.NUM_CLIENTS}")
    
    return extraction_results
print("Full extraction pipeline defined.")

Full extraction pipeline defined.


In [6]:
results = run_full_extraction(config, verbose=True)
print("Full extraction disabled. Uncomment the line above to run.")
print(f"Estimated time: ~{(config.SAMPLES_PER_CLIENT * config.NUM_CLIENTS * 2.2)/60:.0f} minutes")


STEP 1: Loading dataset...
  MildDemented: 8960 images
  ModerateDemented: 6464 images
  NonDemented: 9600 images
  VeryMildDemented: 8960 images

STEP 2: Creating subsets...
Client 1: 2000 images
Client 2: 2000 images
Client 3: 2000 images
Client 4: 2000 images
Client 5: 2000 images

STEP 3: Extracting quantum features...

  Processing Client 1...


Extracting features: 100%|██████████| 2000/2000 [1:14:22<00:00,  2.23s/it]


  -> Saved 2000 features to c:\Users\prakh\Downloads\VIT Downloads\6th Sem\Project\Alzheimer-s-Detection-Using-Quanvolutional-Neural-Networks-with-Federated-Training\quantum_features\client_1
  -> Storage: 31.26 MB
  -> Avg time/image: 2.221s

  Processing Client 2...


Extracting features: 100%|██████████| 2000/2000 [1:14:39<00:00,  2.24s/it]


  -> Saved 2000 features to c:\Users\prakh\Downloads\VIT Downloads\6th Sem\Project\Alzheimer-s-Detection-Using-Quanvolutional-Neural-Networks-with-Federated-Training\quantum_features\client_2
  -> Storage: 31.26 MB
  -> Avg time/image: 2.230s

  Processing Client 3...


...

  -> Saved 2000 features to c:\Users\prakh\Downloads\VIT Downloads\6th Sem\Project\Alzheimer-s-Detection-Using-Quanvolutional-Neural-Networks-with-Federated-Training\quantum_features\client_3
  -> Storage: 31.26 MB
  -> Avg time/image: 2.237s

  Processing Client 4...


Extracting features: 100%|██████████| 2000/2000 [1:18:49<00:00,  2.36s/it]


  -> Saved 2000 features to c:\Users\prakh\Downloads\VIT Downloads\6th Sem\Project\Alzheimer-s-Detection-Using-Quanvolutional-Neural-Networks-with-Federated-Training\quantum_features\client_4
  -> Storage: 31.26 MB
  -> Avg time/image: 2.356s

  Processing Client 5...


Extracting features: 100%|██████████| 2000/2000 [1:17:37<00:00,  2.33s/it]

  -> Saved 2000 features to c:\Users\prakh\Downloads\VIT Downloads\6th Sem\Project\Alzheimer-s-Detection-Using-Quanvolutional-Neural-Networks-with-Federated-Training\quantum_features\client_5
  -> Storage: 31.26 MB
  -> Avg time/image: 2.320s

STEP 4: Saving metadata...

EXTRACTION COMPLETE!
Total time: 380.3 minutes
Output: c:\Users\prakh\Downloads\VIT Downloads\6th Sem\Project\Alzheimer-s-Detection-Using-Quanvolutional-Neural-Networks-with-Federated-Training\quantum_features
Total samples: 10000
Full extraction disabled. Uncomment the line above to run.
Estimated time: ~367 minutes


In [7]:
import numpy as np
import os
from sklearn.model_selection import train_test_split

# Configuration
NUM_CLIENTS = 5
TEST_SIZE = 0.2  # 20% for local testing, 80% for local training
RANDOM_STATE = 42

# Base directory where your client folders are located
BASE_DIR = "quantum_features" 

print(f"Starting local train/test split for {NUM_CLIENTS} clients...")

for i in range(1, NUM_CLIENTS + 1):
    client_name = f"client_{i}"
    client_path = os.path.join(BASE_DIR, client_name)
    
    # 1. Load this specific client's total data
    try:
        features_path = os.path.join(client_path, "features.npy")
        labels_path = os.path.join(client_path, "labels.npy")
        
        X = np.load(features_path)
        y = np.load(labels_path)
        
    except FileNotFoundError:
        print(f"Skipping {client_name}: features.npy or labels.npy not found.")
        continue

    # 2. Split ONLY this client's data
    # Stratify ensures the class balance is maintained in both splits
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, 
        test_size=TEST_SIZE, 
        stratify=y, 
        random_state=RANDOM_STATE
    )

    # 3. Save the splits back to the client's own folder
    np.save(os.path.join(client_path, "train_features.npy"), X_train)
    np.save(os.path.join(client_path, "train_labels.npy"), y_train)
    np.save(os.path.join(client_path, "test_features.npy"), X_test)
    np.save(os.path.join(client_path, "test_labels.npy"), y_test)

    print(f"[{client_name}] Split Complete:")
    print(f"   - Train: {len(X_train)} samples")
    print(f"   - Test:  {len(X_test)} samples")

print("\nAll clients processed.")

Starting local train/test split for 5 clients...
[client_1] Split Complete:
   - Train: 1600 samples
   - Test:  400 samples
[client_2] Split Complete:
   - Train: 1600 samples
   - Test:  400 samples
[client_3] Split Complete:
   - Train: 1600 samples
   - Test:  400 samples
[client_4] Split Complete:
   - Train: 1600 samples
   - Test:  400 samples
[client_5] Split Complete:
   - Train: 1600 samples
   - Test:  400 samples

All clients processed.


In [8]:
#To evaluate the final model, you'll need to combine all test sets:
# In notebook - combine all test data for final evaluation
import numpy as np
all_test_features = []
all_test_labels = []
for i in range(1, 6):
    f = np.load(f"quantum_features/client_{i}/test_features.npy")
    l = np.load(f"quantum_features/client_{i}/test_labels.npy")
    all_test_features.append(f)
    all_test_labels.append(l)
X_test_combined = np.vstack(all_test_features)
y_test_combined = np.concatenate(all_test_labels)
np.save("quantum_features/global_test_features.npy", X_test_combined)
np.save("quantum_features/global_test_labels.npy", y_test_combined)

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np

# Load just ONE client's data
X_train = np.load("quantum_features/client_1/train_features.npy")
y_train = np.load("quantum_features/client_1/train_labels.npy")
X_test = np.load("quantum_features/client_1/test_features.npy")
y_test = np.load("quantum_features/client_1/test_labels.npy")

# Flatten features if they aren't already
if len(X_train.shape) > 2:
    X_train = X_train.reshape(X_train.shape[0], -1)
    X_test = X_test.reshape(X_test.shape[0], -1)

# Fit a basic classifier
clf = RandomForestClassifier(n_estimators=100)
clf.fit(X_train, y_train)
preds = clf.predict(X_test)

print(f"Sanity Check Accuracy: {accuracy_score(y_test, preds):.4f}")

Sanity Check Accuracy: 0.4875


In [10]:
import numpy as np
X = np.load("quantum_features/client_1/train_features.npy")
print(f"Min: {X.min()}, Max: {X.max()}, Mean: {X.mean():.4f}")

Min: -1.0, Max: 0.9999241232872009, Mean: -0.3701


In [11]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
all_features, all_labels = [], []
for i in range(1, 6):
    all_features.append(np.load(f"quantum_features/client_{i}/train_features.npy"))
    all_labels.append(np.load(f"quantum_features/client_{i}/train_labels.npy"))
X = np.vstack(all_features)
y = np.concatenate(all_labels)
# Split uniformly
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=None, random_state=42)
clf = RandomForestClassifier(n_estimators=100)
clf.fit(X_train, y_train)
print(f"Combined RF Accuracy: {accuracy_score(y_test, clf.predict(X_test)):.4f}")

Combined RF Accuracy: 0.5994
